# D4: Analysis

Working through the question end to end: load the flat files, get a feel for what is in them and what is broken, clean them, build district-level metrics, dig into anything that looks odd before trusting it, and test the hypothesis I wrote down in D3.

## Load

In [24]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
for _ in range(8):
    if (ROOT / "requirements.txt").exists():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError("repo root not found: run the notebook from inside the WJCF-takehome repo")
DATA = ROOT / "data_generator" / "data"
OUT = ROOT / "analysis" / "output"
OUT.mkdir(parents=True, exist_ok=True)

claims = pd.read_csv(DATA / "claims.csv", dtype={"district_id": "object", "package_code": "object",
                                                 "date_admitted": "object", "date_discharged": "object"})
facilities = pd.read_csv(DATA / "facilities.csv")
beneficiaries = pd.read_csv(DATA / "beneficiaries.csv")
eligible = pd.read_csv(DATA / "eligible_population.csv")

print("claims:", claims.shape)
print("facilities:", facilities.shape)
print("beneficiaries:", beneficiaries.shape)
print("eligible_population:", eligible.shape)

claims: (13033, 16)
facilities: (178, 10)
beneficiaries: (6314, 6)
eligible_population: (20, 2)


## Step 5a: first pass over the raw data

Before cleaning anything, look at the claims file as it arrives. Note what looks off and how common each problem is, so the cleaning rules and their impact are explicit.

In [25]:
to_date = lambda s: pd.to_datetime(s, errors="coerce")
ad, dd = to_date(claims.date_admitted), to_date(claims.date_discharged)

diagnostics = {
    "duplicate claim_id rows": len(claims) - claims.claim_id.nunique(),
    "discharge before admission": int((dd < ad).sum()),
    "approved_amount > claim_amount": int((claims.approved_amount > claims.claim_amount).sum()),
    "negative claim_amount": int((claims.claim_amount < 0).sum()),
    "implausible beneficiary_age (>110)": int((claims.beneficiary_age > 110).sum()),
    "missing district_id": int(claims.district_id.isna().sum()),
    "missing package_code": int(claims.package_code.isna().sum()),
    "missing date_admitted": int(claims.date_admitted.isna().sum()),
    "missing date_discharged": int(claims.date_discharged.isna().sum()),
}
for k, v in diagnostics.items():
    print(f"{k}: {v}")

bad_hospitals = set(facilities.loc[(facilities.empanelment_status == "de-empanelled") | (facilities.activity_status == "dormant"), "hospital_id"])
print("claims vs de-empanelled or dormant hospitals:", int(claims.hospital_id.isin(bad_hospitals).sum()))

print("\nclaim status distribution:")
print(claims.claim_status.value_counts(normalize=True).round(3))

duplicate claim_id rows: 60
discharge before admission: 30
approved_amount > claim_amount: 60
negative claim_amount: 20
implausible beneficiary_age (>110): 25
missing district_id: 25
missing package_code: 25
missing date_admitted: 20
missing date_discharged: 10
claims vs de-empanelled or dormant hospitals: 819

claim status distribution:
claim_status
approved    0.548
rejected    0.232
pending     0.220
Name: proportion, dtype: float64


## Step 5b: clean

Duplicate rows, missing identifiers and dates, impossible dates, negative amounts, and implausible ages (>110) are to be removed; approved_amount is capped at claim_amount. Each drop is counted so the cleaning is auditable.

In [26]:
c = claims.copy()
c["date_admitted"], c["date_discharged"] = to_date(c.date_admitted), to_date(c.date_discharged)
n0 = len(c)
c = c.drop_duplicates(subset="claim_id", keep="first")
c = c[~c.district_id.isna()]
c = c[~c.package_code.isna()]
c = c[~c.date_admitted.isna() & ~c.date_discharged.isna()]
c = c[c.date_discharged >= c.date_admitted]
c = c[c.claim_amount >= 0]
c["approved_amount"] = np.minimum(c.approved_amount, c.claim_amount)
c = c[c.beneficiary_age <= 110]
c["district_id"] = c.district_id.astype(int)
print(f"clean claims: {n0} -> {len(c)}")

viable = set(facilities.loc[(facilities.empanelment_status == "empanelled") & (facilities.activity_status == "active"), "hospital_id"])
use = c[c.hospital_id.isin(viable)]
use = use[use.claim_status == "approved"]
print("approved claims at empanelled-and-active hospitals:", len(use))

clean claims: 13033 -> 12818
approved claims at empanelled-and-active hospitals: 6577


## Step 5c: two districts show coverage above 100%

Districts 7 and 13 both show more card holders than the eligible population. That is not a number to use at face value. Work out why it happens for each district before deciding how to handle it.

In [27]:
ben_dup = beneficiaries.groupby("district_id")["beneficiary_id"].apply(lambda s: len(s) - s.nunique())
print("duplicate beneficiary rows per district (nonzero only):")
print(ben_dup[ben_dup > 0].to_string())

raw_rows = beneficiaries.groupby("district_id").size().rename("beneficiary_rows")
unique_ids = beneficiaries.groupby("district_id")["beneficiary_id"].nunique().rename("unique_card_holders")
cov = raw_rows.to_frame().join(unique_ids).reset_index().merge(eligible, on="district_id")
cov["raw_coverage"] = cov.beneficiary_rows / cov.eligible_population
cov["clean_coverage"] = cov.unique_card_holders / cov.eligible_population
print("\ncoverage by district (raw vs de-duplicated):")
print(cov.round(3).to_string(index=False))

duplicate beneficiary rows per district (nonzero only):
district_id
7    57

coverage by district (raw vs de-duplicated):
 district_id  beneficiary_rows  unique_card_holders  eligible_population  raw_coverage  clean_coverage
           1               247                  247                  677         0.365           0.365
           2               253                  253                  407         0.622           0.622
           3               266                  266                  362         0.735           0.735
           4               422                  422                  490         0.861           0.861
           5               270                  270                  475         0.568           0.568
           6               162                  162                  464         0.349           0.349
           7               295                  238                  280         1.054           0.850
           8               364                  364   

In [28]:
per_card = (use.groupby("district_id").size() / unique_ids).rename("approved_claims_per_card")
approval_rate = c.groupby("district_id")["claim_status"].apply(lambda s: (s == "approved").mean()).rename("approval_rate")
viable_per_d = facilities[(facilities.empanelment_status == "empanelled") & (facilities.activity_status == "active")].groupby("district_id").size()
per_viable = (use.groupby("district_id").size() / viable_per_d).rename("approved_claims_per_viable_hospital")
peer = pd.concat([per_card, approval_rate, per_viable], axis=1)
print("peer metrics by district (district 11 has no viable hospitals):")
print(peer.round(3).to_string())

print("\npercentile rank of district 13 among peers (0 low, 1 high):")
print(peer.rank(pct=True).loc[13].round(2))

print("\nwhat the checks show:")
print("- District 7 has duplicate beneficiary rows; district 13 has none.")
print("- District 13's claims per card and approval rate sit inside the peer range.")
print("- So its anomaly is in the coverage ratio, not in its claims activity.")

peer metrics by district (district 11 has no viable hospitals):
             approved_claims_per_card  approval_rate  approved_claims_per_viable_hospital
district_id                                                                              
1                               1.259          0.549                               28.273
2                               1.395          0.553                               58.833
3                               1.508          0.555                              401.000
4                               1.149          0.541                               80.833
5                               1.433          0.556                               96.750
6                               1.340          0.582                               21.700
7                               1.004          0.538                               34.143
8                               0.599          0.558                               72.667
9                               0.94

## Step 5d: district metrics

- raw_coverage and clean_coverage: card holders (rows, then de-duplicated IDs) / eligible population
- utilisation per card: approved claims at viable hospitals / de-duplicated card holders
- utilisation per eligible: approved claims / eligible population
- gap: clean_coverage minus utilisation per eligible (reach ahead of use)
- supply per 1000: empanelled-and-active hospitals per 1000 eligible (D1 scale note)
- private share: private share of empanelled-and-active supply

In [29]:
claims_per_district = use.groupby("district_id").size().rename("approved_claims").reset_index()
metrics = cov.merge(claims_per_district, on="district_id", how="left")
metrics["approved_claims"] = metrics.approved_claims.fillna(0).astype(int)
metrics["utilisation_per_card"] = metrics.approved_claims / metrics.unique_card_holders
metrics["utilisation_per_eligible"] = metrics.approved_claims / metrics.eligible_population
metrics["gap"] = metrics.clean_coverage - metrics.utilisation_per_eligible

s = facilities[(facilities.empanelment_status == "empanelled") & (facilities.activity_status == "active")].copy()
s["is_private"] = s.facility_type == "private"
supply_per_district = s.groupby("district_id").agg(supply_count=("hospital_id", "nunique"),
                                                   private_count=("is_private", "sum")).reset_index()
metrics = metrics.merge(supply_per_district, on="district_id", how="left")
metrics["supply_count"] = metrics.supply_count.fillna(0).astype(int)
metrics["private_count"] = metrics.private_count.fillna(0).astype(int)
metrics["supply_per_1000"] = metrics.supply_count / metrics.eligible_population * 1000
metrics["private_share"] = metrics.private_count / metrics.supply_count

keep = ["district_id", "beneficiary_rows", "unique_card_holders", "eligible_population",
        "raw_coverage", "clean_coverage", "approved_claims", "utilisation_per_card",
        "utilisation_per_eligible", "gap", "supply_count", "supply_per_1000", "private_share"]
metrics = metrics[keep].sort_values("gap", ascending=False).reset_index(drop=True)
metrics.to_csv(OUT / "district_metrics.csv", index=False)
print(metrics.round(3).to_string(index=False))

 district_id  beneficiary_rows  unique_card_holders  eligible_population  raw_coverage  clean_coverage  approved_claims  utilisation_per_card  utilisation_per_eligible    gap  supply_count  supply_per_1000  private_share
          11               348                  348                  394         0.883           0.883                0                 0.000                     0.000  0.883             0            0.000            NaN
           8               364                  364                  402         0.905           0.905              218                 0.599                     0.542  0.363             3            7.463          1.000
          20               487                  487                  658         0.740           0.740              401                 0.823                     0.609  0.131             8           12.158          0.625
          17               225                  225                  461         0.488           0.488              

## Step 5e: how much does district 13's position depend on the denominator?

District 13's coverage figure is only as good as the eligible-population estimate behind it. Recompute its coverage and gap under several candidate denominators and see how its position in the gap ranking moves.

In [30]:
ranks = []
for e in [230, 300, 400, 500, 700]:
    tmp = metrics.copy()
    tmp.loc[tmp.district_id == 13, "eligible_population"] = e
    tmp["clean_coverage"] = tmp.unique_card_holders / tmp.eligible_population
    tmp["utilisation_per_eligible"] = tmp.approved_claims / tmp.eligible_population
    tmp["gap"] = tmp.clean_coverage - tmp.utilisation_per_eligible
    ranks.append(int(tmp.sort_values("gap", ascending=False).reset_index(drop=True).query("district_id == 13").index[0]) + 1)

sens = pd.DataFrame({
    "assumed_eligible": [230, 300, 400, 500, 700],
    "coverage": [metrics.loc[metrics.district_id == 13, "unique_card_holders"].iloc[0] / e for e in [230, 300, 400, 500, 700]],
    "supply_per_1000": [metrics.loc[metrics.district_id == 13, "supply_count"].iloc[0] / e * 1000 for e in [230, 300, 400, 500, 700]],
    "district13_gap_rank": ranks,
})
print(sens.round(3).to_string(index=False))
print("\nAt the recorded denominator (230) district 13 ranks", ranks[0], "of", len(metrics), "by gap.")
print("It stays outside the top of the shortlist for every candidate denominator (ranks", min(ranks), "to", max(ranks), "of", len(metrics), ").")
print("So the shortlist does not depend on which denominator assumption is right;")
print("the ratio itself still needs a better denominator before it is quoted.")

 assumed_eligible  coverage  supply_per_1000  district13_gap_rank
              230     1.348           21.739                   16
              300     1.033           16.667                   15
              400     0.775           12.500                   15
              500     0.620           10.000                   13
              700     0.443            7.143                   12

At the recorded denominator (230) district 13 ranks 16 of 20 by gap.
It stays outside the top of the shortlist for every candidate denominator (ranks 12 to 16 of 20 ).
So the shortlist does not depend on which denominator assumption is right;
the ratio itself still needs a better denominator before it is quoted.


## Step 5f: test the hypothesis

The hypothesis I committed to in D3: districts where the empanelled-and-active supply is dominated by private facilities show lower utilisation per card than comparable public-dominated districts. Compare the two groups, controlling for coverage, and see whether the data is consistent with it.

In [32]:
m = metrics.copy()
m["group"] = np.where(m.private_share > 0.5, "private_dominated",
             np.where(m.private_share < 0.5, "public_dominated", "mixed"))
no_supply = m[m.private_share.isna()].copy()
m = m[~m.private_share.isna()].copy()

print("districts excluded from the comparison (no empanelled-active supply):")
print(no_supply[["district_id", "gap", "clean_coverage", "supply_count"]].round(3).to_string(index=False))

means = m.groupby("group").utilisation_per_card.mean()
print("\nmean approved claims per card by group:")
print(means.round(3))
print("difference (private - public):", round(means.get("private_dominated", np.nan) - means.get("public_dominated", np.nan), 3))

print("\nby coverage tercile (control for reach):")
m["coverage_band"] = pd.qcut(m.clean_coverage, 3, labels=["low", "mid", "high"])
pv = m.pivot_table(index="coverage_band", columns="group", values="utilisation_per_card", aggfunc="mean")
print(pv.round(3))

print("\ncorrelation private_share vs utilisation per card:", round(m.private_share.corr(m.utilisation_per_card), 3))
print("correlation supply per 1000 vs gap (all districts with supply):", round(metrics[metrics.private_share.notna()].supply_per_1000.corr(metrics[metrics.private_share.notna()].gap), 3))

districts excluded from the comparison (no empanelled-active supply):
 district_id   gap  clean_coverage  supply_count
          11 0.883           0.883             0

mean approved claims per card by group:
group
mixed                1.033
private_dominated    0.896
public_dominated     1.352
Name: utilisation_per_card, dtype: float64
difference (private - public): -0.456

by coverage tercile (control for reach):
group          mixed  private_dominated  public_dominated
coverage_band                                            
low            1.033              0.913             1.344
mid              NaN              0.923             1.524
high             NaN              0.851             1.187

correlation private_share vs utilisation per card: -0.953
correlation supply per 1000 vs gap (all districts with supply): 0.041


## Findings

- The raw claims file was dirty. Cleaning removed duplicate rows, impossible dates and amounts, implausible ages, missing fields, and claims attached to hospitals that were de-empanelled or dormant. Counts for each are shown above.
- Two districts show more card holders than eligible people, but for different reasons. District 7's number is inflated by duplicate beneficiary rows. District 13 has no duplicates and behaves like its peers on every metric except coverage, so its ratio is best read as a denominator problem: the eligible-population figure looks understated. Under any reasonable denominator it stays out of the priority shortlist, so this does not change which districts to look at first.
- Districts dominated by private hospitals use the scheme less per card than comparable public-dominated districts, and the difference holds when coverage is taken into account. This is consistent with the D3 hypothesis.
- Raw facility counts alone do not explain the gap. The standout cases are district 11, which has no usable hospitals and zero approved claims, and the private-vs-public composition effect among districts that do have supply. The defensible takeaway is about who owns the hospitals, not how many there are.